# Dimensión de cuentas

Construye `data/interim/dimensiones/dim_cuentas.parquet` a partir del plan de cuentas del BCRA (`cuentas.txt`) del dump IEF de corte diciembre 2025.

La tabla expone una fila por código de cuenta con denominación, fecha de baja (si aplica), nivel jerárquico y flags para regularizadoras y cuentas CERA. Se usa para joinear contra los paneles de balance cuando hace falta saber qué representa cada código.

Decisiones metodológicas aplicadas (ver `docs/notas/metodologia_paneles.md` §3.2, §3.8): las regularizadoras quedan identificadas con un flag pero no se consolidan en esta etapa; `fecha_baja` se preserva como viene del BCRA.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

CUENTAS_RAW = RAW / "bcra_ief/202601/Entfin/Tec_Cont/cuentas/cuentas.txt"
OUT = DIMENSIONES / "dim_cuentas.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

## Lectura

El archivo es TSV con tres campos: código, denominación y fecha de baja. Codificación ISO-8859-1, saltos CRLF. La denominación preserva los espacios iniciales que codifican el nivel jerárquico.

In [ ]:
raw = pd.read_csv(
    CUENTAS_RAW,
    sep="\t",
    header=None,
    names=["codigo_cuenta", "denominacion_raw", "fecha_baja_raw"],
    encoding="ISO-8859-1",
    dtype=str,
)
print(f"Filas leídas: {len(raw):,}")
raw.head()

## Parseo

`nivel` sale de contar los espacios iniciales en la denominación. La fecha de baja `"/  /"` indica cuenta vigente y se convierte a NaT; el resto son fechas `DD/MM/YYYY`.

In [ ]:
raw["nivel"] = raw["denominacion_raw"].str.len() - raw["denominacion_raw"].str.lstrip().str.len()
raw["denominacion"] = raw["denominacion_raw"].str.strip()

fecha_str = raw["fecha_baja_raw"].str.strip().replace({"/  /": None, "/ /": None, "//": None, "": None})
raw["fecha_baja"] = pd.to_datetime(fecha_str, format="%d/%m/%Y", errors="coerce")

raw[["codigo_cuenta", "denominacion", "fecha_baja", "nivel"]].head(10)

## Flags derivados

`es_regularizadora`: cuentas cuya denominación empieza con `(`. Convención BCRA para marcar cuentas que restan del capítulo madre (incluye regularizadoras propiamente dichas, previsiones por incobrabilidad, por desvalorización, etc.). Ver metodología §3.2.

`es_cera`: las cuatro cuentas introducidas por la Com. "A" 8071 para el régimen Ley 27.743 (`311793`, `312183`, `315794`, `316147`).

In [ ]:
raw["es_regularizadora"] = raw["denominacion"].str.startswith("(")
raw["es_cera"] = raw["denominacion"].str.contains(r"27\.?743", regex=True, na=False)

raw.loc[raw["es_cera"], ["codigo_cuenta", "denominacion"]]

## Persistencia

In [ ]:
dim = raw[["codigo_cuenta", "denominacion", "fecha_baja", "nivel", "es_regularizadora", "es_cera"]].copy()
dim.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(dim):,}")

## Validaciones

In [ ]:
assert len(dim) == 5519, f"Número de filas inesperado: {len(dim)}"
assert dim["codigo_cuenta"].is_unique, "Hay códigos duplicados"
assert dim["codigo_cuenta"].str.len().eq(6).all(), "Hay códigos que no son de 6 dígitos"
assert dim["es_cera"].sum() == 4, f"Se esperan 4 cuentas CERA, hay {dim['es_cera'].sum()}"
assert dim["es_regularizadora"].sum() >= 100, "Conteo de regularizadoras más bajo que lo esperado"

resumen = {
    "vigentes": int(dim["fecha_baja"].isna().sum()),
    "discontinuadas": int(dim["fecha_baja"].notna().sum()),
    "regularizadoras": int(dim["es_regularizadora"].sum()),
    "cera": int(dim["es_cera"].sum()),
}
print("Validaciones OK")
for k, v in resumen.items():
    print(f"  {k}: {v:,}")

## Muestra con DuckDB

In [ ]:
duckdb.sql(f"""
    select codigo_cuenta, denominacion, fecha_baja, nivel
    from '{OUT}'
    where es_cera
    order by codigo_cuenta
""").df()